# Витрина транзакций ТЭ — 8. Карточка клиента

Всё по одному клиенту: кто он, итог тестов тетрадки 07, точки и терминалы, операции по месяцам, по часам и по суммам в сравнении с его группой, повторяющиеся суммы, карты, статусы, ночные операции.

Как пользоваться: впишите ИНН в ячейку ниже и запустите все ячейки (Run All). Нужны собранные 01–05 и файл `te_client_tests_2026_08_10.xlsx` из тетрадки 07 в этой же папке. Без файла не будет итога тестов и нормы группы, остальное посчитается.

Период — всё, что лежит в факте: август–октябрь 2026.

In [ ]:
from IPython.display import display
from rail_connectors.connection import connect

imp = connect(
    to="IMPALA",
    extra_options={"db": "sandbox_ai"},
    driver_args={"tez.queue.name": "ai"},
    kerberos={
        "keytab_path": "/home/jovyan/test_requests/tech.keytab",
        "use_credentials": True,
        "update_keytab": True,
    },
    user_params={"user_name": "Shestopalov-VYur"},
)
imp._init_connection()

In [ ]:
from pathlib import Path

import pandas as pd

INN = ""

INN = "".join(ch for ch in str(INN) if ch.isdigit())
assert INN, "Впишите ИНН клиента в INN"
pd.options.display.float_format = lambda x: f"{x:,.0f}".replace(",", " ") if abs(x) >= 1000 else f"{x:.3f}"
pd.options.display.max_colwidth = 300
TESTS_FILE = Path.cwd() / "te_client_tests_2026_08_10.xlsx"
TRX = "sandbox_ai.tmp_shestopalov_acq_trx"


def q(sql):
    with imp:
        return imp.fetch(sql)


def pct(frame, cols):
    out = frame.copy()
    for c in cols:
        out[c] = (out[c] * 100).round(1).astype(str) + "%"
    return out

## Кто клиент

Карточка из `tmp_shestopalov_acq_clients` и профиль за период из `tmp_shestopalov_te_profile`, показаны вертикально.

In [ ]:
client = q(f"select * from sandbox_ai.tmp_shestopalov_acq_clients where inn = '{INN}'")
profile = q(f"select * from sandbox_ai.tmp_shestopalov_te_profile where inn = '{INN}'")
if client.empty and profile.empty:
    raise ValueError(f"ИНН {INN} нет ни в карточке, ни в профиле — проверьте номер")
display(client.T.rename(columns={0: "карточка"}))
display(profile.T.rename(columns={0: "профиль"}))

## Итог тестов тетрадки 07

Строки клиента с листа `все_проверки`: по одной на каждую его группу. Флаги, сила отличия и пояснение словами. Ниже — была ли строка на листе `подозрительные`.

In [ ]:
checks = norms = None
if TESTS_FILE.exists():
    checks = pd.read_excel(TESTS_FILE, sheet_name="все_проверки", dtype={"inn": str, "группа": str})
    norms = pd.read_excel(TESTS_FILE, sheet_name="норма_групп", dtype={"группа": str})
    suspicious = pd.read_excel(TESTS_FILE, sheet_name="подозрительные", dtype={"inn": str})
    mine = checks.loc[checks["inn"] == INN]
    if mine.empty:
        print("Клиент в тестах не участвовал: меньше 30 операций или все его группы меньше 30 клиентов.")
    else:
        cols = ["разрез", "группа_название", "клиентов_в_группе", "операций", "ks_d", "mw_доля_пар", "chi_v",
                "косинус", "флаг_ks", "флаг_mw", "флаг_chi", "флаг_cos", "флагов", "ночь_доля",
                "ночь_доля_группы", "ночь", "главный_час", "что_выбивается"]
        display(mine[[c for c in cols if c in mine.columns]])
    print("На листе «подозрительные»:", "да" if (suspicious["inn"] == INN).any() else "нет")
else:
    print(f"Нет файла {TESTS_FILE.name} — запустите тетрадку 07 или положите файл в эту папку.")

## Точки и терминалы

По каждому терминалу: точка (`retl_id`), город точки, код точки по профилю терминала (`mcc_terminal`) и код в чеке (`mcc_code`), операции, оборот, первая и последняя операция, модель устройства, даты установки и снятия. Несколько городов или кодов — повод спросить, где клиент торгует на самом деле.

In [ ]:
points = q(f"""
select
  t.id_pos_terminal, t.retl_id, t.mrc_city, t.mcc_terminal, t.mcc_profile, t.mcc_code,
  count(*) as ops,
  sum(case when t.status <> 'R' then t.sum_t else 0 end) as turnover_ok,
  min(t.date_t) as first_op,
  max(t.date_t) as last_op,
  max(d.version) as device,
  max(d.date_on) as date_on,
  max(d.date_off) as date_off
from {TRX} t
left join sandbox_ai.tmp_shestopalov_acq_terminals d
  on d.id_pos_terminal = t.id_pos_terminal
where t.inn = '{INN}'
group by t.id_pos_terminal, t.retl_id, t.mrc_city, t.mcc_terminal, t.mcc_profile, t.mcc_code
order by ops desc
""")
display(points)

## По месяцам

Операции, оборот успешных, средний чек, доля отменённых (статус `R`) и активные дни по месяцам. Резкий рост или провал — повод уточнить.

In [ ]:
months = q(f"""
select
  trunc(date_t, 'MM') as month,
  count(*) as ops,
  sum(case when status <> 'R' then sum_t else 0 end) as turnover_ok,
  avg(case when status <> 'R' then sum_t end) as avg_check,
  avg(case when status = 'R' then 1.0 else 0.0 end) as share_r,
  count(distinct to_date(date_t)) as active_days
from {TRX}
where inn = '{INN}'
group by trunc(date_t, 'MM')
order by month
""")
display(pct(months, ["share_r"]))

## По часам: клиент против группы

Доля операций клиента в каждом часе (без отменённых) и та же доля в норме его группы по коду точки с листа `норма_групп`. Колонка `разница` — на сколько пунктов клиент выше группы. Часы 0–5 — ночь.

Если пик клиента приходится на 04–07, а у группы на 11–13, проверьте часовой пояс: у терминала время может писаться по Москве. Разница региона с Москвой — в ячейке проверки сдвига в тетрадке 07.

In [ ]:
hours = q(f"""
select cast(hour(date_t) as int) as hour, count(*) as ops
from {TRX}
where inn = '{INN}' and status <> 'R'
group by cast(hour(date_t) as int)
""").set_index("hour").reindex(range(24), fill_value=0)
hours["доля_клиента"] = hours["ops"] / hours["ops"].sum()

mcc = str(profile["mcc_code"].iloc[0]) if not profile.empty else None
if norms is not None and mcc is not None:
    row = norms.loc[(norms["разрез"] == "MCC") & (norms["группа"] == mcc)]
    if not row.empty:
        hours["доля_группы"] = [float(row[f"час_{h:02d}"].iloc[0]) for h in range(24)]
        hours["разница"] = hours["доля_клиента"] - hours["доля_группы"]
        print("Группа:", row["группа_название"].iloc[0], "| клиентов:", int(row["клиентов"].iloc[0]))
share_cols = [c for c in ("доля_клиента", "доля_группы", "разница") if c in hours.columns]
display(pct(hours, share_cols))
print(f"Ночь 0–6 ч: {hours['доля_клиента'].iloc[:6].sum() * 100:.1f}% операций клиента"
      + (f" против {hours['доля_группы'].iloc[:6].sum() * 100:.1f}% в группе" if "доля_группы" in hours else ""))

## По суммам: клиент против группы

Те же 18 корзин суммы чека, что в тестах. Доля клиента, доля нормы группы по коду точки и разница.

In [ ]:
AMOUNT_LABELS = [
    "до 100", "100-200", "200-300", "300-500", "500-700", "700-1000",
    "1000-1500", "1500-2000", "2000-3000", "3000-5000", "5000-7000", "7000-10000",
    "10-15 тыс", "15-20 тыс", "20-30 тыс", "30-50 тыс", "50-100 тыс", "от 100 тыс",
]
EDGES = [100, 200, 300, 500, 700, 1000, 1500, 2000, 3000, 5000, 7000, 10000, 15000, 20000, 30000, 50000, 100000]
case = "\n".join(f"      when sum_t < {e} then {i}" for i, e in enumerate(EDGES))
amount = q(f"""
select bin, count(*) as ops
from (
  select cast(case
{case}
      else {len(EDGES)}
    end as int) as bin
  from {TRX}
  where inn = '{INN}' and status <> 'R'
) a
group by bin
""").set_index("bin").reindex(range(len(AMOUNT_LABELS)), fill_value=0)
amount.index = AMOUNT_LABELS
amount["доля_клиента"] = amount["ops"] / amount["ops"].sum()
if norms is not None and mcc is not None:
    row = norms.loc[(norms["разрез"] == "MCC") & (norms["группа"] == mcc)]
    if not row.empty:
        amount["доля_группы"] = [float(row[f"чек_{lab}"].iloc[0]) for lab in AMOUNT_LABELS]
        amount["разница"] = amount["доля_клиента"] - amount["доля_группы"]
display(pct(amount, [c for c in ("доля_клиента", "доля_группы", "разница") if c in amount.columns]))

## Повторяющиеся суммы

Самые частые суммы чека. Если большая доля операций приходится на одну-две круглые суммы, это похоже на перевод денег через терминал, а не на торговлю.

In [ ]:
repeat = q(f"""
select sum_t, count(*) as ops
from {TRX}
where inn = '{INN}' and status <> 'R'
group by sum_t
order by ops desc
limit 15
""")
total_ok = int(hours["ops"].sum())
repeat["доля_операций"] = repeat["ops"] / total_ok
display(pct(repeat, ["доля_операций"]))

## Карты и статусы

Самые частые BIN карт (первые цифры номера, банк и тип карты) и платёжные системы. Если почти все операции идут с нескольких BIN, покупателей мало, и это стоит уточнить. Ниже — статусы и типы операций.

In [ ]:
bins = q(f"""
select card_bin, payment_system, count(*) as ops, sum(sum_t) as amount
from {TRX}
where inn = '{INN}' and status <> 'R'
group by card_bin, payment_system
order by ops desc
limit 15
""")
bins["доля_операций"] = bins["ops"] / total_ok
display(pct(bins, ["доля_операций"]))

statuses = q(f"""
select status, type_t, count(*) as ops, sum(sum_t) as amount
from {TRX}
where inn = '{INN}'
group by status, type_t
order by ops desc
""")
display(statuses)

## Ночные операции

Последние 50 операций с 0 до 6 часов: когда, на какую сумму, через какой терминал и с какой карты (только BIN).

In [ ]:
night = q(f"""
select date_t, sum_t, status, type_t, id_pos_terminal, retl_id, mrc_city, mcc_code, card_bin, payment_system
from {TRX}
where inn = '{INN}' and hour(date_t) < 6
order by date_t desc
limit 50
""")
print(f"Ночных операций в выборке: {len(night)}")
display(night)

## По дням недели

Распределение операций по дням недели (1 — воскресенье, 7 — суббота, как в Impala). Магазин, который работает только в выходные или только по будням, бывает законным, но это стоит знать.

In [ ]:
days = q(f"""
select dayofweek(date_t) as dow, count(*) as ops, sum(sum_t) as amount
from {TRX}
where inn = '{INN}' and status <> 'R'
group by dayofweek(date_t)
order by dow
""")
days["доля"] = days["ops"] / days["ops"].sum()
display(pct(days, ["доля"]))